In [11]:
import pandas as pd
import numpy as np
import random
import os

# reproducibilidad
np.random.seed(42)
random.seed(42)

NUM_RECORDS = 500

# Demográficos
ages = np.random.randint(17, 30, size=NUM_RECORDS)
genders = np.random.choice(['Male', 'Female', 'Non-Binary'], size=NUM_RECORDS, p=[0.48, 0.48, 0.04])
places_of_origin = np.random.choice(
    ['Urban', 'Rural', 'Suburban', 'International'],
    size=NUM_RECORDS,
    p=[0.6, 0.25, 0.1, 0.05]
)

# Académicos
hs_grades = np.random.normal(loc=80, scale=10, size=NUM_RECORDS).clip(50, 100)
admission_test = np.random.normal(loc=75, scale=12, size=NUM_RECORDS).clip(0, 100)

# relación entre notas (no totalmente aleatorio)
first_sem_grades = 0.6 * hs_grades + 0.3 * admission_test + np.random.normal(0, 10, NUM_RECORDS)
first_sem_grades = first_sem_grades.clip(0, 100)

# Financieros
socioeconomic_levels = np.random.choice(['Low', 'Middle', 'High'], size=NUM_RECORDS, p=[0.4, 0.5, 0.1])
scholarships = np.random.choice(['Yes', 'No'], size=NUM_RECORDS, p=[0.3, 0.7])
loans = np.random.choice(['Yes', 'No'], size=NUM_RECORDS, p=[0.4, 0.6])

# dropout: cálculo del riesgo
risk_score = np.zeros(NUM_RECORDS)
risk_score += np.where(first_sem_grades < 60, 2, 0)
risk_score += np.where((socioeconomic_levels == 'Low') & (scholarships == 'No'), 1.5, 0)
risk_score += np.where(places_of_origin == 'Rural', 0.5, 0)
risk_score += np.where(admission_test < 50, 1, 0)
risk_score += np.where(loans == 'Yes', 0.5, 0)

# ruido para que no sea perfecto
risk_score += np.random.normal(0, 1, NUM_RECORDS)

dropouts = np.where(risk_score > 2.5, 'Yes', 'No')

# DataFrame
df = pd.DataFrame({
    'Student_ID': range(1, NUM_RECORDS + 1),
    'Age': ages,
    'Gender': genders,
    'Place_of_Origin': places_of_origin,
    'HS_Grade_Avg': hs_grades,
    'Admission_Test_Score': admission_test,
    'First_Semester_Grade': first_sem_grades,
    'Socioeconomic_Level': socioeconomic_levels,
    'Scholarship': scholarships,
    'Financial_Loan': loans,
    'Dropout': dropouts
})

# nulls
df.loc[np.random.choice(df.index, int(NUM_RECORDS * 0.05), replace=False), 'First_Semester_Grade'] = np.nan
df.loc[np.random.choice(df.index, int(NUM_RECORDS * 0.025), replace=False), 'Financial_Loan'] = np.nan

# outliers
df.loc[np.random.choice(df.index, 5, replace=False), 'HS_Grade_Avg'] = np.array([150, 180, 200, 195, 175], dtype=float)
df.loc[np.random.choice(df.index, 3, replace=False), 'Age'] = np.array([5, 12, 65], dtype=np.int32)

# formato
df['HS_Grade_Avg'] = df['HS_Grade_Avg'].round(2)
df['Admission_Test_Score'] = df['Admission_Test_Score'].round(2)
df['First_Semester_Grade'] = df['First_Semester_Grade'].round(2)

# guardar csv
output_path = "student_dropout_dataset.csv"
df.to_csv(output_path, index=False)